In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

c:\Users\Tanvi\OneDrive\Desktop\ATS_SCORER\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load the cleaned ATS pairs dataset
df = pd.read_csv('../dataset/cleaned_resumeJD_pairs.csv')
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['match_label'].value_counts())
df.head(3)

Loaded: 280 rows, 7 columns
Columns: ['resume_text', 'job_description', 'match_label', 'match_score', 'match_label_lower', 'resume_len', 'jd_len']

Label distribution:
match_label
medium    110
high       93
low        77
Name: count, dtype: int64


,resume_text,job_description,match_label,match_score,match_label_lower,resume_len,jd_len
0,BI Analyst with 7 years of experience based in...,Fathom is hiring a Financial Analyst (Berlin)....,medium,0.53,medium,75,71
1,Data Scientist with 13 years of experience bas...,Lumen is hiring a Senior UX Designer (Singapor...,low,0.26,low,79,74
2,Software Engineer with 8 years of experience b...,Acme Corp is hiring a Product Manager (Seattle...,low,0.31,low,71,80


In [6]:
print("loading the BERT model")
model = SentenceTransformer('all-mpnet-base-v2')

print('Model loaded!')
print(f"Model: all-mpnet-base-v2")
print(f"parameters: 22.7M")
print(f"Embed dims: 768")
print(f"Max Tokens: {model.max_seq_length}")

loading the BERT model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 337.33it/s]


Model loaded!
Model: all-mpnet-base-v2
parameters: 22.7M
Embed dims: 768
Max Tokens: 384


In [7]:
# generating resume and JD embeddings
# we embed both resume and job description seperately

print("Generating resume embeddings....")
resume_embeddings = model.encode(
    df["resume_text"].to_list(),
    batch_size= 32,
    show_progress_bar=True,
    convert_to_numpy= True
)

print("\nGenerating job description embeddings....")
jd_embeddings = model.encode(
    df["job_description"].to_list(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)


print(f"\nResume embeddings shape: {resume_embeddings.shape}")
print(f"JD embeddings shape: {jd_embeddings.shape}")
print(f"Each text -> {resume_embeddings.shape[1]} - dimensional vector")

Generating resume embeddings....


Batches: 100%|██████████| 9/9 [00:31<00:00,  3.47s/it]



Generating job description embeddings....


Batches: 100%|██████████| 9/9 [00:24<00:00,  2.74s/it]


Resume embeddings shape: (280, 768)
JD embeddings shape: (280, 768)
Each text -> 768 - dimensional vector


In [9]:
# compute cosine similarity for each pair
# for each row : similarity between that resume and its paired JD

pair_similarities = []
for i in range(len(resume_embeddings)):
    sim = cosine_similarity([resume_embeddings[i]], [jd_embeddings[i]])[0][0]
    pair_similarities.append(sim)

df['bert_similarity'] = pair_similarities

print('Cosine similarity computed for all pairs.')
print(df[['match_label', 'match_score', 'bert_similarity']].head(10))

Cosine similarity computed for all pairs.
  match_label  match_score  bert_similarity
0      medium         0.53         0.554188
1         low         0.26         0.489453
2         low         0.31         0.471077
3      medium         0.46         0.620440
4        high         0.77         0.660347
5        high         0.92         0.608102
6         low         0.13         0.515025
7      medium         0.45         0.619569
8      medium         0.46         0.562903
9      medium         0.48         0.479238


In [ ]:
# 